# Strategy 3: hybrid MonopolyZero training on Colab CPU

Runs `Strategy 3`'s pipeline: PPO warm start -> decaying ASU bootstrap (teacher
role) -> self-play (+ ASU as a low-probability opponent seat) -> seat-balanced
evaluation against Fixed-A/B/C.

Read `Strategy 3/PLAN.md` first. Three decisions this notebook assumes, already
settled there:

- **Metric (PLAN.md §1)**: baseline-relative. The number that matters is this
  candidate's seat-balanced win rate against Fixed-A/B/C, compared against the
  measured `asu_value_v1` baseline of **72/100** (`Strategy 1/REPO_STUDY_NOTES.md`).
  ASU is never seated as an opponent in the final evaluation cell.
- **ASU's role (PLAN.md §2)**: Option B. ASU occupies a training opponent seat
  at low probability, AND supplies a decaying bootstrap policy target before
  self-play starts (never during self-play itself -- self-play only learns from
  real game outcomes, per `CLAUDE.md`'s "never train on ASU's output" rule).
- Heavy training belongs here (Colab), not local -- `Strategy 3/CLAUDE.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DeepRL_Monopoly'
!mkdir -p "$DRIVE_ROOT/artifacts"
!mkdir -p "$DRIVE_ROOT/monopoly_bench_runs"

## Get the code

Clones from the `Gokturkakman/DeepRL_Monopoly` fork (no write access to
`Darkosxl/DeepRL_Monopoly` upstream), branch `feature/strategy-3-hybrid` --
`Strategy 3` hasn't been merged to `main` yet. If you get a 404 or a tree
without `Strategy 3/`, either the branch was renamed/merged since this was
written (update `BRANCH` below) or it hasn't been pushed yet.

In [ ]:
REPO_URL = 'https://github.com/Gokturkakman/DeepRL_Monopoly.git'  # fork -- no write access to Darkosxl/DeepRL_Monopoly
BRANCH = 'feature/strategy-3-hybrid'  # not merged to main yet

!rm -rf /content/DeepRL_Monopoly
!git clone --branch $BRANCH --depth 1 $REPO_URL /content/DeepRL_Monopoly
%cd "/content/DeepRL_Monopoly/Strategy 3"
!ls

In [ ]:
# Colab ships torch preinstalled; this just confirms the CPU build works
# and numpy is present. No CUDA wheel needed -- this pipeline is CPU-only.
!pip install -q --upgrade numpy
import torch, numpy
print('torch', torch.__version__, 'cuda available (unused here):', torch.cuda.is_available())
print('numpy', numpy.__version__)

## Step 1 -- PPO warm start

`monopoly_bench`'s bootstrap step (Step 2) initializes its actor/policy head
from a PPO checkpoint (`MonopolyZeroNet.load_ppo_actor`). Train that checkpoint
first. `--asu-opponent-probability 0.02` adds ASU as a training opponent seat
(PLAN.md §2, Option B's opponent-seat half) -- kept low because ASU's own
decision cost is 1-2 orders of magnitude above a fixed heuristic's. This never
records ASU's chosen actions as a training target; only the game outcome
feeds back into PPO (`train.py`'s `_sample_opponents`).

In [ ]:
PPO_OUT = f'{DRIVE_ROOT}/artifacts/ppo_plus/ppo_hybrid_2000_v2.pt'

!python tools/train_and_save.py \
  --algo ppo --hybrid \
  --games 2000 \
  --device cpu \
  --seed 42 \
  --checkpoint-every 100 \
  --opponent-epsilon 0.1 \
  --opponent-threshold-jitter 0.2 \
  --held-out-eval-games 20 \
  --asu-opponent-probability 0.02 \
  --out "$PPO_OUT"

(Optional, separate track) Train DDQN with the diagnosed short-run fix --
default `lr=1e-5` / `target_update_freq=500 games` are tuned for a 10,000-game
paper run; a 1000-game run at those defaults stayed at 0% win rate with
correctly-signed rewards throughout (`CLAUDE.md`, "Known-hard problem"). This
DDQN run is independent of the MonopolyZero pipeline below -- run it if you
also want a DDQN baseline number, skip it if you only want the hybrid track.

In [ ]:
DDQN_OUT = f'{DRIVE_ROOT}/artifacts/ddqn_plus/ddqn_hybrid_colab.pt'

!python tools/train_and_save.py \
  --algo ddqn --hybrid \
  --games 2000 \
  --device cpu \
  --seed 42 \
  --checkpoint-every 100 \
  --lr 1e-4 \
  --target-update-freq-steps 2000 \
  --epsilon-decay 0.9985 \
  --opponent-epsilon 0.1 \
  --opponent-threshold-jitter 0.2 \
  --held-out-eval-games 20 \
  --asu-opponent-probability 0.02 \
  --out "$DDQN_OUT"

## Step 2 -- ASU bootstrap dataset (teacher role, decaying)

`collect-asu` plays ASU-vs-ASU games and records its chosen actions as a
one-time offline dataset. This is the *only* place ASU's decisions are ever
recorded as a training target, and it is not self-play -- `monopoly_bench`
decays this dataset's influence to zero over `expert_decay_generations`
(8, `config.py`), so by the final generations training runs purely on real
game outcomes. This does not violate `CLAUDE.md`'s "never train PPO/DDQN/CFR/
MonopolyZero self-play on ASU's output" rule -- the rule targets self-play,
this step precedes it.

In [ ]:
ASU_EXPERT_OUT = f'{DRIVE_ROOT}/monopoly_bench_runs/asu_expert.npz'

!python -m monopoly_bench collect-asu \
  --output "$ASU_EXPERT_OUT" \
  --games 256 \
  --seed-base 100000

## Step 3 -- self-play generations

`Trainer.run(generations)` warm-starts from the PPO actor (Step 1), applies
the decaying ASU bootstrap (Step 2) for early generations, then self-play
(against snapshots + Fixed-A-F + a low-probability ASU opponent seat, all
outcome-only) for the rest. `--generations` here is intentionally small for a
first Colab pass -- raise it once a short run shows the win rate moving.
Checkpoints every generation live under `RUN_DIR` on Drive, so a disconnect
loses at most the in-progress generation.

In [ ]:
RUN_DIR = f'{DRIVE_ROOT}/monopoly_bench_runs/strategy3_run1'

!python -m monopoly_bench train \
  --run-dir "$RUN_DIR" \
  --bootstrap-ppo "$PPO_OUT" \
  --asu-expert-data "$ASU_EXPERT_OUT" \
  --generations 4 \
  --device cpu

Resume more generations later (same `--run-dir`, `Trainer` picks up from the
last checkpointed generation automatically):

In [ ]:
!python -m monopoly_bench train \
  --run-dir "$RUN_DIR" \
  --bootstrap-ppo "$PPO_OUT" \
  --asu-expert-data "$ASU_EXPERT_OUT" \
  --generations 12 \
  --device cpu

## Step 4 -- the metric that matters (PLAN.md §1)

Seat-balanced win rate against Fixed-A/B/C, with a Wilson lower bound
(`tools/evaluate_vs_fixed.py`, built on `monopoly_bench.arena`). Compare
`win_rate` directly against the measured `asu_value_v1` baseline of **72/100**
(`Strategy 1/REPO_STUDY_NOTES.md` §1). ASU is not an opponent in this cell --
per PLAN.md §1 this is a baseline-relative comparison, not a head-to-head
against ASU.

In [ ]:
import glob

# Prefer a promoted snapshot (passed the league gate); fall back to the
# latest candidate if nothing has been promoted yet -- still a real
# save_inference checkpoint, just not gate-cleared.
candidates = sorted(glob.glob(f'{RUN_DIR}/snapshots/promoted_*.pt'))
if not candidates:
    candidates = sorted(glob.glob(f'{RUN_DIR}/candidates/generation_*.pt'))
if not candidates:
    raise SystemExit('No evaluable checkpoint yet -- run Step 3 first.')
CANDIDATE = candidates[-1]
print('Evaluating:', CANDIDATE)

!python tools/evaluate_vs_fixed.py \
  --candidate "$CANDIDATE" \
  --games 100 \
  --out "$RUN_DIR/eval_vs_fixed_abc_100.json"

Checkpoints, the ASU expert dataset, run directories, and the eval JSON all
live under `$DRIVE_ROOT` on Drive, so they survive the Colab VM being
recycled. Pull anything back to your laptop by downloading it from Drive
directly -- don't route it through git; `artifacts/` and `monopoly_bench_runs/`
are gitignored on purpose.